In [1]:
# ============================================================
# CELL 1 — INSTALL & ENVIRONMENT
# ============================================================
!pip install ultralytics -q
!pip install kornia -q        # GPU-accelerated Sobel + Gabor filters

import os, gc, yaml, shutil, random, math, json
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import kornia
import kornia.filters as KF
from collections import defaultdict
from torch.utils.data import Dataset, DataLoader
from ultralytics import YOLO
from pathlib import Path
from PIL import Image
import torchvision.transforms.functional as TF
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# ── Reproducibility ──────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ── DDP env vars for Kaggle dual-T4 ──────────────────────────
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['NCCL_P2P_DISABLE']   = '1'
os.environ['NCCL_IB_DISABLE']    = '1'

gc.collect()
torch.cuda.empty_cache()

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")
for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print(f"  GPU{i}: {torch.cuda.get_device_name(i)} | {free/1e9:.1f} GB free / {total/1e9:.1f} GB total")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 6.3 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Device: cuda
  GPU0: Tesla T4 | 15.5 GB free / 15.6 GB total
  GPU1: Tesla T4 | 15.5 GB free / 15.6 GB total


In [2]:
# ============================================================
# CELL 2 — DATASET MERGE & STRATIFIED SPLIT (FIXED)
# ============================================================
import os, shutil, random
from collections import defaultdict

# ✅ Correct dataset path
src_base = '/kaggle/input/datasets/josephinenakalembe/weld-defect-detection'

# 🔍 Auto-detect actual root (handles nested Kaggle folders)
def find_dataset_root(base):
    for root, dirs, _ in os.walk(base):
        if 'train' in dirs:
            return root
    return base

src = find_dataset_root(src_base)
print(f"Using dataset root: {src}")

dst = '/kaggle/working/dataset_merged'

# 🔍 Detect whether dataset uses 'val' or 'valid'
splits_available = os.listdir(src)
val_split_name = 'valid' if 'valid' in splits_available else 'val'

print(f"Detected splits: {splits_available}")
print(f"Validation folder name: {val_split_name}")

if not os.path.exists(f'{dst}/train/images'):
    
    # Create destination folders
    for split in ['train','val','test']:
        os.makedirs(f'{dst}/{split}/images', exist_ok=True)
        os.makedirs(f'{dst}/{split}/labels', exist_ok=True)

    all_images = []

    # ✅ Use detected split names
    for split in ['train', val_split_name, 'test']:
        img_dir = f'{src}/{split}/images'
        lbl_dir = f'{src}/{split}/labels'

        if not os.path.exists(img_dir):
            print(f"Skipping missing: {img_dir}")
            continue

        for f in os.listdir(img_dir):
            if f.lower().endswith(('.jpg','.png','.jpeg')):
                lbl = f.rsplit('.',1)[0] + '.txt'
                lbl_path = f'{lbl_dir}/{lbl}'
                if os.path.exists(lbl_path):
                    all_images.append((f'{img_dir}/{f}', lbl_path))

    print(f"Total images found: {len(all_images)}")

    # 🚫 Stop early if still broken
    if len(all_images) == 0:
        raise ValueError("No images found. Check dataset structure.")

    def get_dominant_class(lbl_path):
        counts = defaultdict(int)
        with open(lbl_path) as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    counts[int(parts[0])] += 1
        return max(counts, key=counts.get) if counts else -1

    random.seed(42)
    random.shuffle(all_images)

    class_buckets = defaultdict(list)
    for img_path, lbl_path in all_images:
        cls = get_dominant_class(lbl_path)
        class_buckets[cls].append((img_path, lbl_path))

    train_set, val_set, test_set = [], [], []

    for cls, items in class_buckets.items():
        random.shuffle(items)
        n       = len(items)
        n_train = int(n * 0.75)
        n_val   = int(n * 0.125)

        train_set.extend(items[:n_train])
        val_set.extend(items[n_train:n_train+n_val])
        test_set.extend(items[n_train+n_val:])

    def copy_split(items, split):
        for img_path, lbl_path in items:
            shutil.copy(img_path, f'{dst}/{split}/images/{os.path.basename(img_path)}')
            shutil.copy(lbl_path, f'{dst}/{split}/labels/{os.path.basename(lbl_path)}')

    copy_split(train_set, 'train')
    copy_split(val_set,   'val')
    copy_split(test_set,  'test')

    print(f"Train: {len(train_set)} | Val: {len(val_set)} | Test: {len(test_set)}")

else:
    print("Dataset already exists ✓")


# ============================================================
# YAML FILE
# ============================================================
import yaml

yaml_path = '/kaggle/working/data_merged.yaml'

CLASS_NAMES = ['Crack', 'Porosity', 'Spatters', 'Welding line']

with open(yaml_path, 'w') as f:
    yaml.dump({
        'path':  dst,
        'train': 'train/images',
        'val':   'val/images',
        'test':  'test/images',
        'nc':    4,
        'names': CLASS_NAMES,
    }, f, default_flow_style=False, sort_keys=False)

print("YAML ready ✓")

Using dataset root: /kaggle/input/datasets/josephinenakalembe/weld-defect-detection/1
Detected splits: ['README.dataset.txt', 'README.roboflow.txt', 'data.yaml', 'valid', 'test', 'train']
Validation folder name: valid
Total images found: 6420
Train: 4813 | Val: 800 | Test: 807
YAML ready ✓


In [3]:
# ============================================================
# CELL 3 — STAGE 1: PSEUDO-MODALITY GENERATOR (FIXED)
# ============================================================

class PseudoModalityGenerator(nn.Module):

    def __init__(self,
                 gabor_kernel_size: int = 15,
                 gabor_orientations: int = 4):
        super().__init__()

        # ── ImageNet normalization ───────────────────────────
        self.register_buffer('mean', torch.tensor([0.485, 0.456, 0.406]).view(1,3,1,1))
        self.register_buffer('std',  torch.tensor([0.229, 0.224, 0.225]).view(1,3,1,1))

        # ── Gabor filter bank ────────────────────────────────
        self.gabor_orientations = gabor_orientations
        gabor_kernels = self._build_gabor_bank(
            kernel_size=gabor_kernel_size,
            n_orientations=gabor_orientations
        )
        # (n_orient, 1, k, k)
        self.register_buffer('gabor_kernels', gabor_kernels)

        # ── Fuse orientations → 1 channel ────────────────────
        self.gabor_fuse = nn.Conv2d(gabor_orientations, 1, kernel_size=1, bias=False)
        nn.init.constant_(self.gabor_fuse.weight, 1.0 / gabor_orientations)

    # ─────────────────────────────────────────────────────────
    def _build_gabor_bank(self, kernel_size, n_orientations):
        kernels = []
        for i in range(n_orientations):
            theta = i * math.pi / n_orientations
            k = self._gabor_kernel(kernel_size, 3.0, theta, 6.0, 0.5, 0.0)
            kernels.append(k.unsqueeze(0).unsqueeze(0))
        return torch.cat(kernels, dim=0)

    def _gabor_kernel(self, kernel_size, sigma, theta, lambd, gamma, psi):
        half = kernel_size // 2
        y, x = torch.meshgrid(
            torch.arange(-half, half+1, dtype=torch.float32),
            torch.arange(-half, half+1, dtype=torch.float32),
            indexing='ij'
        )
        x_theta =  x * math.cos(theta) + y * math.sin(theta)
        y_theta = -x * math.sin(theta) + y * math.cos(theta)

        gaussian = torch.exp(-0.5 * (x_theta**2 + (gamma**2) * y_theta**2) / sigma**2)
        sinusoid = torch.cos(2 * math.pi * x_theta / lambd + psi)

        kernel = gaussian * sinusoid
        kernel = kernel - kernel.mean()
        return kernel

    # ─────────────────────────────────────────────────────────
    def forward(self, x):

        # ── Grayscale ────────────────────────────────────────
        gray = (0.299 * x[:,0:1] + 0.587 * x[:,1:2] + 0.114 * x[:,2:3])

        # ── GRADIENT MAP (Sobel) ─────────────────────────────
        grad = kornia.filters.spatial_gradient(gray)
        gx = grad[:, :, 0]
        gy = grad[:, :, 1]

        gradient_map = torch.sqrt(gx**2 + gy**2 + 1e-8)

        gmin = gradient_map.flatten(2).min(dim=2)[0].unsqueeze(-1).unsqueeze(-1)
        gmax = gradient_map.flatten(2).max(dim=2)[0].unsqueeze(-1).unsqueeze(-1)
        gradient_map = (gradient_map - gmin) / (gmax - gmin + 1e-8)

        # ── TEXTURE MAP (FIXED Gabor) ────────────────────────
        B, _, H, W = gray.shape
        pad = self.gabor_kernels.shape[-1] // 2

        # Repeat input channels for grouped conv
        gray_rep = gray.repeat(1, self.gabor_orientations, 1, 1)

        gabor_resp = F.conv2d(
            gray_rep,
            self.gabor_kernels,
            padding=pad,
            groups=self.gabor_orientations
        )  # (B, n_orient, H, W)

        gabor_resp = torch.abs(gabor_resp)

        texture_map = self.gabor_fuse(gabor_resp)
        texture_map = F.relu(texture_map)

        tmin = texture_map.flatten(2).min(dim=2)[0].unsqueeze(-1).unsqueeze(-1)
        tmax = texture_map.flatten(2).max(dim=2)[0].unsqueeze(-1).unsqueeze(-1)
        texture_map = (texture_map - tmin) / (tmax - tmin + 1e-8)

        # ── RGB NORMALIZED ───────────────────────────────────
        rgb_norm = (x - self.mean) / self.std

        return gradient_map, texture_map, rgb_norm


# ── Sanity check ─────────────────────────────────────────────
_gen = PseudoModalityGenerator().to(DEVICE)
_x   = torch.rand(2, 3, 640, 640).to(DEVICE)

_g, _t, _r = _gen(_x)

print("[Stage 1] PseudoModalityGenerator ✓")
print(f"gradient_map : {_g.shape}  min={_g.min():.3f} max={_g.max():.3f}")
print(f"texture_map  : {_t.shape}  min={_t.min():.3f} max={_t.max():.3f}")
print(f"rgb_norm     : {_r.shape}")

[Stage 1] PseudoModalityGenerator ✓
gradient_map : torch.Size([2, 1, 640, 640])  min=0.000 max=1.000
texture_map  : torch.Size([2, 1, 640, 640])  min=0.000 max=1.000
rgb_norm     : torch.Size([2, 3, 640, 640])


In [4]:
# ============================================================
# CELL 4 — STAGE 2: PARALLEL ENCODERS (FIXED)
# ============================================================

import torch
import torch.nn as nn

# ────────────────────────────────────────────────────────────
class ConvBnRelu(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3,
                      stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)


class FeatureEncoder(nn.Module):
    def __init__(self, in_channels: int, out_channels: int = 256):
        super().__init__()
        self.encoder = nn.Sequential(
            ConvBnRelu(in_channels,  64,  stride=2),
            ConvBnRelu(64,          128, stride=2),
            ConvBnRelu(128, out_channels, stride=2),
        )

    def forward(self, x):
        return self.encoder(x)


class ParallelEncoders(nn.Module):
    def __init__(self, feat_dim: int = 256):
        super().__init__()

        self.encoder_g = FeatureEncoder(1, feat_dim)
        self.encoder_t = FeatureEncoder(1, feat_dim)
        self.encoder_r = FeatureEncoder(3, feat_dim)

    def forward(self, gradient_map, texture_map, rgb_norm):
        F_g = self.encoder_g(gradient_map)
        F_t = self.encoder_t(texture_map)
        F_r = self.encoder_r(rgb_norm)
        return F_g, F_t, F_r


# ── Sanity check ─────────────────────────────────────────────
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# create dummy inputs if previous cell not run
if '_g' not in globals():
    print("⚠️ Using dummy inputs (Stage 1 not found)")
    _g = torch.rand(2, 1, 640, 640).to(DEVICE)
    _t = torch.rand(2, 1, 640, 640).to(DEVICE)
    _r = torch.rand(2, 3, 640, 640).to(DEVICE)

_enc = ParallelEncoders(feat_dim=256).to(DEVICE)

_Fg, _Ft, _Fr = _enc(_g, _t, _r)

print("[Stage 2] ParallelEncoders ✓")
print(f"F_gradient : {_Fg.shape}")
print(f"F_texture  : {_Ft.shape}")
print(f"F_RGB      : {_Fr.shape}")

total_params = sum(p.numel() for p in _enc.parameters())
print(f"Total encoder params: {total_params:,}")

[Stage 2] ParallelEncoders ✓
F_gradient : torch.Size([2, 256, 80, 80])
F_texture  : torch.Size([2, 256, 80, 80])
F_RGB      : torch.Size([2, 256, 80, 80])
Total encoder params: 1,111,488


In [5]:
# ============================================================
# SAFE FULL PIPELINE TEST (self-contained)
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import kornia

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# ────────────────────────────────────────────────────────────
# DEFINE STAGE 1 IF MISSING
# ────────────────────────────────────────────────────────────
if 'PseudoModalityGenerator' not in globals():

    class PseudoModalityGenerator(nn.Module):
        def __init__(self, gabor_kernel_size=15, gabor_orientations=4):
            super().__init__()

            self.register_buffer('mean', torch.tensor([0.485,0.456,0.406]).view(1,3,1,1))
            self.register_buffer('std',  torch.tensor([0.229,0.224,0.225]).view(1,3,1,1))

            self.gabor_orientations = gabor_orientations
            self.gabor_kernels = self._build_gabor_bank(gabor_kernel_size, gabor_orientations)

            self.gabor_fuse = nn.Conv2d(gabor_orientations, 1, 1, bias=False)
            nn.init.constant_(self.gabor_fuse.weight, 1.0 / gabor_orientations)

        def _build_gabor_bank(self, k, n):
            kernels = []
            for i in range(n):
                theta = i * math.pi / n
                kernels.append(self._gabor_kernel(k, 3.0, theta, 6.0, 0.5, 0.0).unsqueeze(0).unsqueeze(0))
            return torch.cat(kernels, dim=0)

        def _gabor_kernel(self, k, sigma, theta, lambd, gamma, psi):
            half = k // 2
            y, x = torch.meshgrid(torch.arange(-half, half+1), torch.arange(-half, half+1), indexing='ij')
            y, x = y.float(), x.float()

            x_t = x*math.cos(theta) + y*math.sin(theta)
            y_t = -x*math.sin(theta) + y*math.cos(theta)

            g = torch.exp(-0.5*(x_t**2 + (gamma**2)*y_t**2)/sigma**2)
            s = torch.cos(2*math.pi*x_t/lambd + psi)

            k = g*s
            return k - k.mean()

        def forward(self, x):
            gray = (0.299*x[:,0:1] + 0.587*x[:,1:2] + 0.114*x[:,2:3])

            grad = kornia.filters.spatial_gradient(gray)
            gx, gy = grad[:,:,0], grad[:,:,1]
            g = torch.sqrt(gx**2 + gy**2 + 1e-8)

            g = (g - g.min()) / (g.max() - g.min() + 1e-8)

            pad = self.gabor_kernels.shape[-1] // 2
            gray_rep = gray.repeat(1, self.gabor_orientations, 1, 1)

            t = F.conv2d(gray_rep, self.gabor_kernels.to(x.device),
                         padding=pad, groups=self.gabor_orientations)
            t = torch.abs(t)

            t = self.gabor_fuse(t)
            t = F.relu(t)
            t = (t - t.min()) / (t.max() - t.min() + 1e-8)

            r = (x - self.mean) / self.std
            return g, t, r


# ────────────────────────────────────────────────────────────
# DEFINE STAGE 2 IF MISSING
# ────────────────────────────────────────────────────────────
if 'ParallelEncoders' not in globals():

    class ConvBnRelu(nn.Module):
        def __init__(self, in_ch, out_ch, stride=1):
            super().__init__()
            self.block = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 3, stride, 1, bias=False),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True)
            )
        def forward(self, x):
            return self.block(x)

    class FeatureEncoder(nn.Module):
        def __init__(self, in_ch):
            super().__init__()
            self.net = nn.Sequential(
                ConvBnRelu(in_ch, 64, 2),
                ConvBnRelu(64, 128, 2),
                ConvBnRelu(128, 256, 2),
            )
        def forward(self, x):
            return self.net(x)

    class ParallelEncoders(nn.Module):
        def __init__(self):
            super().__init__()
            self.g = FeatureEncoder(1)
            self.t = FeatureEncoder(1)
            self.r = FeatureEncoder(3)

        def forward(self, g, t, r):
            return self.g(g), self.t(t), self.r(r)


# ────────────────────────────────────────────────────────────
# RUN FULL PIPELINE
# ────────────────────────────────────────────────────────────
gen = PseudoModalityGenerator().to(DEVICE)
enc = ParallelEncoders().to(DEVICE)

x = torch.rand(2, 3, 640, 640).to(DEVICE)

g, t, r = gen(x)
Fg, Ft, Fr = enc(g, t, r)

print("✅ Pipeline OK")
print(Fg.shape, Ft.shape, Fr.shape)

✅ Pipeline OK
torch.Size([2, 256, 80, 80]) torch.Size([2, 256, 80, 80]) torch.Size([2, 256, 80, 80])


In [6]:
# ============================================================
# CELL 5 — STAGE 3: CRAG FUSION MODULE (CORE INNOVATION)
# ============================================================
#
# Step 1: Cross-Attention (6 Bidirectional Paths)
#   Gradient ↔ Texture : "Do edges align with texture patterns?"
#   Gradient ↔ RGB     : "What is the spatial context of these edges?"
#   Texture  ↔ RGB     : "Where are these texture patterns located?"
#
# Step 2: Adaptive Weight Generation
#   Concat(F_g, F_t, F_r) → Conv3×3 → Conv1×1 → Softmax
#   → [W_g(i,j), W_t(i,j), W_r(i,j)]  where W_g + W_t + W_r = 1
#
# Step 3: Adaptive Fusion
#   F_fused(i,j) = W_g⊙F_g + W_t⊙F_t + W_r⊙F_r
#
# Expected learned behaviour (from your proposal):
#   Crack     → W_g ≈ 0.70  (trust sharp gradient features)
#   Porosity  → W_t ≈ 0.65  (trust irregular texture)
#   Spatter   → balanced    (W_g≈0.35, W_t≈0.40, W_r≈0.25)
#   Weld Line → W_r ≈ 0.60  (trust RGB spatial context)
# ============================================================

class CrossAttentionGate(nn.Module):
    """
    Bidirectional cross-attention between two feature streams A and B.

    Implements: A' = A + sigmoid(Conv1x1(B)) ⊙ A
                B' = B + sigmoid(Conv1x1(A)) ⊙ B

    Intuition: B acts as a "gate" that highlights relevant regions of A,
    and A simultaneously gates B. This lets each stream ask:
    "Which parts of the other stream are relevant to me?"

    Uses channel-wise compression for efficiency (ratio=8).
    """
    def __init__(self, feat_dim: int, reduction: int = 8):
        super().__init__()
        mid_ch = max(feat_dim // reduction, 16)  # compressed channel dim

        # A → gate for B
        self.gate_A2B = nn.Sequential(
            nn.Conv2d(feat_dim, mid_ch,   kernel_size=1, bias=False),
            nn.BatchNorm2d(mid_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_ch,   feat_dim, kernel_size=1, bias=False),
            nn.Sigmoid()
        )
        # B → gate for A
        self.gate_B2A = nn.Sequential(
            nn.Conv2d(feat_dim, mid_ch,   kernel_size=1, bias=False),
            nn.BatchNorm2d(mid_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_ch,   feat_dim, kernel_size=1, bias=False),
            nn.Sigmoid()
        )

    def forward(self, A, B):
        """
        Args:
            A, B: (B, C, H, W) — feature maps from two streams
        Returns:
            A_attended, B_attended: cross-attended feature maps
        """
        gate_for_A = self.gate_B2A(B)       # B tells A "look here"
        gate_for_B = self.gate_A2B(A)       # A tells B "look here"
        A_attended = A + gate_for_A * A     # residual: keep original + gated
        B_attended = B + gate_for_B * B
        return A_attended, B_attended


class CRAGFusionModule(nn.Module):
    """
    STAGE 3 — Cross-Representation Attention Gate Fusion.

    Inputs:
        F_g, F_t, F_r : (B, 256, H/8, W/8)
    Output:
        F_fused       : (B, 256, H/8, W/8)  — class-adaptive fused features
        weights       : (B, 3,   H/8, W/8)  — attention weights for visualisation
    """
    def __init__(self, feat_dim: int = 256):
        super().__init__()
        self.feat_dim = feat_dim

        # ── Step 1: 3 Cross-Attention Gates (6 bidirectional paths) ──
        # Path 1&2: Gradient ↔ Texture
        self.ca_gt = CrossAttentionGate(feat_dim)
        # Path 3&4: Gradient ↔ RGB
        self.ca_gr = CrossAttentionGate(feat_dim)
        # Path 5&6: Texture ↔ RGB
        self.ca_tr = CrossAttentionGate(feat_dim)

        # ── Step 2: Adaptive Weight Generator ────────────────
        # Concat(F_g, F_t, F_r) = (B, 768, H/8, W/8)
        # → Conv3×3 → 256ch  (local context aggregation)
        # → Conv1×1 → 3ch    (one weight map per stream)
        # → Softmax over 3ch (spatial, per-pixel normalisation)
        self.weight_gen = nn.Sequential(
            nn.Conv2d(feat_dim * 3, feat_dim, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(feat_dim),
            nn.ReLU(inplace=True),
            nn.Conv2d(feat_dim, 3, kernel_size=1, bias=True),
            # NOTE: Softmax applied in forward() over dim=1 (the 3-stream dim)
        )

        # ── Optional: Final projection after fusion ───────────
        # Refine the fused representation with a small residual block
        self.refine = nn.Sequential(
            nn.Conv2d(feat_dim, feat_dim, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(feat_dim),
            nn.ReLU(inplace=True)
        )

    def forward(self, F_g, F_t, F_r):
        """
        Args:
            F_g : (B, 256, Hf, Wf)  gradient features
            F_t : (B, 256, Hf, Wf)  texture  features
            F_r : (B, 256, Hf, Wf)  RGB      features
        Returns:
            F_fused : (B, 256, Hf, Wf)
            weights : (B, 3, Hf, Wf)  [W_g, W_t, W_r] — for visualisation
        """
        # ── Step 1: Cross-attention ───────────────────────────
        # Each gate enriches both streams with information from the other
        F_g_gt, F_t_gt = self.ca_gt(F_g, F_t)   # gradient-texture dialogue
        F_g_gr, F_r_gr = self.ca_gr(F_g, F_r)   # gradient-RGB dialogue
        F_t_tr, F_r_tr = self.ca_tr(F_t, F_r)   # texture-RGB dialogue

        # Aggregate attended versions (residual sum across paths)
        # Each stream benefits from BOTH cross-attention paths it participates in
        F_g_att = F_g + F_g_gt + F_g_gr         # gradient: enriched by texture + RGB
        F_t_att = F_t + F_t_gt + F_t_tr         # texture:  enriched by gradient + RGB
        F_r_att = F_r + F_r_gr + F_r_tr         # RGB:      enriched by gradient + texture

        # ── Step 2: Adaptive weight generation ───────────────
        # Concatenate all three attended feature maps
        F_cat = torch.cat([F_g_att, F_t_att, F_r_att], dim=1)  # (B, 768, Hf, Wf)
        raw_weights = self.weight_gen(F_cat)                     # (B, 3, Hf, Wf)

        # Softmax over the 3-stream dimension → W_g + W_t + W_r = 1 everywhere
        weights = F.softmax(raw_weights, dim=1)                  # (B, 3, Hf, Wf)
        W_g = weights[:, 0:1, :, :]    # (B, 1, Hf, Wf)
        W_t = weights[:, 1:2, :, :]    # (B, 1, Hf, Wf)
        W_r = weights[:, 2:3, :, :]    # (B, 1, Hf, Wf)

        # ── Step 3: Adaptive fusion ───────────────────────────
        # F_fused(i,j) = W_g(i,j) ⊙ F_g(i,j)
        #               + W_t(i,j) ⊙ F_t(i,j)
        #               + W_r(i,j) ⊙ F_r(i,j)
        F_fused = (W_g * F_g_att) + (W_t * F_t_att) + (W_r * F_r_att)

        # Optional refinement convolution
        F_fused = self.refine(F_fused)                           # (B, 256, Hf, Wf)

        return F_fused, weights


# ── Quick sanity check ────────────────────────────────────────
if True:
    _crag = CRAGFusionModule(feat_dim=256).to(DEVICE)
    _Ff, _W = _crag(_Fg, _Ft, _Fr)
    print("[Stage 3] CRAGFusionModule ✓")
    print(f"  F_fused : {_Ff.shape}")
    print(f"  weights : {_W.shape}  [W_g, W_t, W_r]")
    print(f"  Weight sum check (should be 1.0): {_W.sum(dim=1).mean():.4f}")
    _wmean = _W.mean(dim=(0,2,3))  # mean weight per stream (batch avg)
    print(f"  Mean weights → W_g={_wmean[0]:.3f}  W_t={_wmean[1]:.3f}  W_r={_wmean[2]:.3f}")
    total_crag = sum(p.numel() for p in _crag.parameters())
    print(f"  CRAG module params: {total_crag:,}")

[Stage 3] CRAGFusionModule ✓
  F_fused : torch.Size([2, 256, 80, 80])
  weights : torch.Size([2, 3, 80, 80])  [W_g, W_t, W_r]
  Weight sum check (should be 1.0): 1.0000
  Mean weights → W_g=0.340  W_t=0.326  W_r=0.334
  CRAG module params: 2,459,779


In [7]:
# ============================================================
# CELL 6 — CRAGNET PREPROCESSOR (Stages 1–3 Combined)
# ============================================================
# Takes a raw RGB image batch (B, 3, H, W)
# Runs Stage 1 → Stage 2 → Stage 3
# Outputs a fused feature map upsampled back to (B, 3, H, W)
# so that YOLOv11 receives an input in the same shape and
# channel count it expects — no YOLO architecture changes needed.
# ============================================================

class CRAGNetPreprocessor(nn.Module):
    """
    Complete CRAG-Net front-end (Stages 1, 2, 3).

    Input  : (B, 3, H, W)   — raw RGB images in [0, 1]
    Output : (B, 3, H, W)   — CRAG-fused representation (upsampled)
             plus optional side outputs for analysis:
             gradient_map, texture_map, weights

    Why upsample back to (B, 3, H, W)?
    YOLOv11 expects 3-channel input at the full resolution.
    We project the 256-channel fused features (at H/8) back to
    3-channel full resolution using a transposed conv + bilinear
    upsample. The result is an enriched 3-channel image.
    """

    def __init__(self,
                 feat_dim: int = 256,
                 gabor_kernel_size: int = 15,
                 gabor_orientations: int = 4):
        super().__init__()

        # Stage 1
        self.stage1 = PseudoModalityGenerator(
            gabor_kernel_size=gabor_kernel_size,
            gabor_orientations=gabor_orientations
        )

        # Stage 2
        self.stage2 = ParallelEncoders(feat_dim=feat_dim)

        # Stage 3
        self.stage3 = CRAGFusionModule(feat_dim=feat_dim)

        # ── Decoder: fused (B,256,H/8,W/8) → (B,3,H,W) ──────
        # Two upsampling steps (each ×2) + final ×2 bilinear upsample
        # followed by a learned projection to 3 channels
        self.decoder = nn.Sequential(
            # (B, 256, H/8, W/8) → (B, 128, H/4, W/4)
            nn.ConvTranspose2d(feat_dim, 128, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            # (B, 128, H/4, W/4) → (B, 64, H/2, W/2)
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            # (B, 64, H/2, W/2) → (B, 3, H/2, W/2)
            nn.Conv2d(64, 3, kernel_size=3, padding=1, bias=True),
        )

        # ── Residual mixing scalar ────────────────────────────
        # output = alpha * crag_output + (1 - alpha) * rgb_input
        # Initialised to 0.5 so training starts from a balanced blend
        # and learns how much to trust the CRAG representation
        self.alpha = nn.Parameter(torch.tensor(0.5))

    def forward(self, x, return_intermediates=False):
        """
        Args:
            x: (B, 3, H, W)  — raw RGB in [0, 1]
            return_intermediates: if True, also return gradient_map,
                                  texture_map, weights for analysis
        Returns:
            out: (B, 3, H, W)  — CRAG-enhanced image for YOLO
            [optional] gradient_map, texture_map, weights
        """
        B, C, H, W = x.shape

        # ── Stage 1: Pseudo-modality generation ──────────────
        gradient_map, texture_map, rgb_norm = self.stage1(x)

        # ── Stage 2: Parallel feature extraction ─────────────
        F_g, F_t, F_r = self.stage2(gradient_map, texture_map, rgb_norm)

        # ── Stage 3: CRAG fusion ──────────────────────────────
        F_fused, weights = self.stage3(F_g, F_t, F_r)

        # ── Decode fused features back to image resolution ────
        crag_out = self.decoder(F_fused)         # (B, 3, H/2, W/2)
        crag_out = F.interpolate(                # (B, 3, H, W)
            crag_out, size=(H, W),
            mode='bilinear', align_corners=False
        )

        # ── Residual blend: combine with original input ───────
        # Clamp alpha to valid range during training
        alpha = self.alpha.clamp(0.0, 1.0)
        out = alpha * crag_out + (1.0 - alpha) * rgb_norm

        if return_intermediates:
            return out, gradient_map, texture_map, weights
        return out


# ── Sanity check ──────────────────────────────────────────────
if True:
    _prep = CRAGNetPreprocessor(feat_dim=256).to(DEVICE)
    _x_raw = torch.rand(2, 3, 640, 640).to(DEVICE)
    _out, _gm, _tm, _wts = _prep(_x_raw, return_intermediates=True)
    print("[Combined] CRAGNetPreprocessor ✓")
    print(f"  Input  shape : {_x_raw.shape}")
    print(f"  Output shape : {_out.shape}")
    print(f"  Alpha (blend): {_prep.alpha.item():.3f}")
    total_prep = sum(p.numel() for p in _prep.parameters())
    trainable  = sum(p.numel() for p in _prep.parameters() if p.requires_grad)
    print(f"  Total params    : {total_prep:,}")
    print(f"  Trainable params: {trainable:,}")
    print(f"  GPU memory used : {torch.cuda.memory_allocated()/1e9:.2f} GB")

[Combined] CRAGNetPreprocessor ✓
  Input  shape : torch.Size([2, 3, 640, 640])
  Output shape : torch.Size([2, 3, 640, 640])
  Alpha (blend): 0.500
  Total params    : 4,228,747
  Trainable params: 4,228,747
  GPU memory used : 2.49 GB


In [8]:
# ============================================================
# CELL 7 — ATTENTION WEIGHT VISUALISER
# ============================================================
# Generates a 6-panel figure per image:
#   [Original | Gradient Map | Texture Map | W_g | W_t | W_r]
# This is directly useful for your thesis Section 4.2 analysis
# (mirrors the Grad-CAM analysis in the Wang et al. paper).
# ============================================================

@torch.no_grad()
def visualise_crag_attention(preprocessor, image_path, device=DEVICE, save_path=None):
    """
    Visualise CRAG attention weights for a single image.

    Args:
        preprocessor : trained CRAGNetPreprocessor
        image_path   : path to a weld image (.jpg / .png)
        device       : torch device
        save_path    : if given, save figure instead of showing it
    """
    preprocessor.eval()

    # Load and preprocess image
    img = Image.open(image_path).convert('RGB').resize((640, 640))
    x   = TF.to_tensor(img).unsqueeze(0).to(device)  # (1,3,640,640)

    # Forward pass with intermediates
    out, grad_map, tex_map, weights = preprocessor(x, return_intermediates=True)

    # Move to CPU for plotting
    orig     = x[0].cpu().permute(1, 2, 0).numpy()
    grad_np  = grad_map[0, 0].cpu().numpy()
    tex_np   = tex_map[0, 0].cpu().numpy()
    W_g = weights[0, 0].cpu().numpy()   # gradient weight map
    W_t = weights[0, 1].cpu().numpy()   # texture  weight map
    W_r = weights[0, 2].cpu().numpy()   # RGB      weight map

    # Upsample weight maps from H/8 to H for display
    def up(m):
        t = torch.tensor(m).unsqueeze(0).unsqueeze(0)
        return F.interpolate(t, size=(640, 640), mode='bilinear',
                             align_corners=False)[0, 0].numpy()
    W_g, W_t, W_r = up(W_g), up(W_t), up(W_r)

    # ── Plot ─────────────────────────────────────────────────
    fig, axes = plt.subplots(2, 3, figsize=(15, 9))
    fig.suptitle('CRAG-Net Attention Analysis', fontsize=14, fontweight='bold')

    titles = ['Original Image', 'Gradient Map (Sobel)',
              'Texture Map (Gabor)',
              'W_gradient (Crack-sensitive)',
              'W_texture  (Porosity-sensitive)',
              'W_RGB      (WeldLine-sensitive)']
    images = [orig, grad_np, tex_np, W_g, W_t, W_r]
    cmaps  = [None, 'hot', 'hot', 'RdYlGn', 'RdYlGn', 'RdYlGn']

    for ax, title, img_data, cmap in zip(axes.flat, titles, images, cmaps):
        if cmap is None:
            ax.imshow(np.clip(img_data, 0, 1))
        else:
            im = ax.imshow(img_data, cmap=cmap, vmin=0, vmax=1)
            plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        ax.set_title(title, fontsize=10)
        ax.axis('off')

    # Print mean attention weights
    print(f"Mean attention weights for this image:")
    print(f"  W_gradient : {W_g.mean():.3f}  (high → crack-like content)")
    print(f"  W_texture  : {W_t.mean():.3f}  (high → porosity-like content)")
    print(f"  W_RGB      : {W_r.mean():.3f}  (high → welding-line context)")

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Figure saved: {save_path}")
    else:
        plt.show()
    plt.close()


@torch.no_grad()
def analyse_class_weights(preprocessor, dataset_dir, device=DEVICE, n_per_class=20):
    """
    Compute average attention weights per defect class across the dataset.
    Expected result (from your proposal):
      Crack     → high W_g
      Porosity  → high W_t
      Spatter   → balanced
      Weld Line → high W_r
    This is the key interpretability result for your thesis.
    """
    preprocessor.eval()
    class_names = ['Crack', 'Porosity', 'Spatters', 'Welding line']
    class_weights = {name: [] for name in class_names}

    img_dir = Path(dataset_dir) / 'test' / 'images'
    lbl_dir = Path(dataset_dir) / 'test' / 'labels'

    class_buckets = defaultdict(list)
    for img_path in sorted(img_dir.glob('*.jpg'))[:500]:  # limit scan
        lbl_path = lbl_dir / (img_path.stem + '.txt')
        if not lbl_path.exists():
            continue
        with open(lbl_path) as f:
            lines = [l.strip().split() for l in f if l.strip()]
        if not lines:
            continue
        cls = int(lines[0][0])
        class_buckets[cls].append(img_path)

    for cls_idx, cls_name in enumerate(class_names):
        paths = class_buckets[cls_idx][:n_per_class]
        wg_list, wt_list, wr_list = [], [], []
        for p in paths:
            img = Image.open(p).convert('RGB').resize((640, 640))
            x   = TF.to_tensor(img).unsqueeze(0).to(device)
            _, _, _, weights = preprocessor(x, return_intermediates=True)
            wg_list.append(weights[0, 0].mean().item())
            wt_list.append(weights[0, 1].mean().item())
            wr_list.append(weights[0, 2].mean().item())
        if wg_list:
            print(f"{cls_name:15s}: W_g={np.mean(wg_list):.3f}  "
                  f"W_t={np.mean(wt_list):.3f}  W_r={np.mean(wr_list):.3f}")

print("Visualiser utilities defined ✓")

Visualiser utilities defined ✓


In [9]:
# ============================================================
# CELL 8 — CRAG-YOLOv11 WRAPPER
# ============================================================
# Strategy: Use Ultralytics' model callbacks to inject the
# CRAG preprocessor into the training loop without modifying
# the YOLO source code.
#
# How it works:
#   on_train_start  → attach CRAGNetPreprocessor to model
#   on_train_batch_start → run images through CRAG before YOLO sees them
#   on_val_batch_start   → same for validation
#
# The CRAG parameters are added to the optimiser via
# on_optimizer_step so they get gradients from YOLO's loss.
# ============================================================

class CRAGCallbackHandler:
    """
    Ultralytics callback handler that plugs CRAGNetPreprocessor
    into the YOLOv11 training loop.

    Usage:
        model   = YOLO('yolo11s.pt')
        handler = CRAGCallbackHandler(feat_dim=256)
        handler.register(model)
        model.train(...)   # CRAG trains end-to-end with YOLO
    """

    def __init__(self, feat_dim=256, device=None):
        self.feat_dim = feat_dim
        self.device   = device or ('cuda' if torch.cuda.is_available() else 'cpu')
        self.preprocessor = None
        self.crag_optimizer = None

    def register(self, yolo_model):
        """Register all callbacks with the YOLO model."""
        yolo_model.add_callback('on_train_start',       self.on_train_start)
        yolo_model.add_callback('on_train_batch_start', self.on_train_batch_start)
        yolo_model.add_callback('on_val_batch_start',   self.on_val_batch_start)
        yolo_model.add_callback('on_train_epoch_end',   self.on_train_epoch_end)
        print("CRAG callbacks registered ✓")

    # ── Callback: initialise CRAG at training start ───────────
    def on_train_start(self, trainer):
        print("[CRAG] Initialising CRAGNetPreprocessor...")
        self.preprocessor = CRAGNetPreprocessor(
            feat_dim=self.feat_dim
        ).to(self.device)

        # Separate optimiser for CRAG parameters
        # Uses same lr as YOLO optimiser for consistency
        base_lr = trainer.args.lr0 if hasattr(trainer.args, 'lr0') else 1e-3
        self.crag_optimizer = torch.optim.AdamW(
            self.preprocessor.parameters(),
            lr=base_lr,
            weight_decay=5e-4
        )

        total = sum(p.numel() for p in self.preprocessor.parameters())
        print(f"[CRAG] Preprocessor ready | params={total:,} | device={self.device}")

    # ── Callback: transform batch images before YOLO forward ──
    def on_train_batch_start(self, trainer):
        if self.preprocessor is None:
            return
        self.preprocessor.train()
        batch = trainer.batch
        imgs  = batch['img'].to(self.device).float() / 255.0  # [0,1]
        # Run CRAG — gradients flow back through preprocessor
        self.crag_optimizer.zero_grad()
        crag_out = self.preprocessor(imgs)
        # Replace batch images with CRAG output (scaled back to [0,255])
        batch['img'] = (crag_out * 255.0).clamp(0, 255)

    # ── Callback: transform validation images (no grad) ───────
    def on_val_batch_start(self, validator):
        if self.preprocessor is None:
            return
        self.preprocessor.eval()
        batch = validator.batch
        with torch.no_grad():
            imgs    = batch['img'].to(self.device).float() / 255.0
            crag_out = self.preprocessor(imgs)
            batch['img'] = (crag_out * 255.0).clamp(0, 255)

    # ── Callback: step CRAG optimiser after each epoch ────────
    def on_train_epoch_end(self, trainer):
        if self.crag_optimizer is not None:
            self.crag_optimizer.step()

    def save(self, path):
        """Save CRAG preprocessor weights separately."""
        if self.preprocessor:
            torch.save(self.preprocessor.state_dict(), path)
            print(f"[CRAG] Saved preprocessor weights → {path}")

    def load(self, path):
        """Load previously saved CRAG weights."""
        if self.preprocessor is None:
            self.preprocessor = CRAGNetPreprocessor(feat_dim=self.feat_dim).to(self.device)
        self.preprocessor.load_state_dict(torch.load(path, map_location=self.device))
        print(f"[CRAG] Loaded preprocessor weights ← {path}")


print("CRAGCallbackHandler defined ✓")

CRAGCallbackHandler defined ✓


In [10]:
# ============================================================
# CELL 9 — MODEL A: YOLOV11 BASELINE (unchanged from original)
# ============================================================
TRAIN_BASELINE = False   # ← Set True to (re)train Model A

if TRAIN_BASELINE:
    last_ckpt = '/kaggle/working/weld_defect_detection/yolo11s_baseline/weights/last.pt'
    if os.path.exists(last_ckpt):
        model_A = YOLO(last_ckpt)
        resume  = True
        print(f"Resuming Model A from: {last_ckpt}")
    else:
        model_A = YOLO('yolo11s.pt')
        resume  = False
        print("Training Model A from scratch")

    model_A.train(
        data=yaml_path, epochs=100, imgsz=640, batch=32,
        device=[0, 1], workers=4, amp=True, resume=resume,
        hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
        degrees=10, translate=0.1, scale=0.5,
        shear=2.0, fliplr=0.5, mosaic=1.0,
        mixup=0.1, copy_paste=0.1,
        optimizer='AdamW', lr0=0.001, lrf=0.01, cos_lr=True,
        weight_decay=0.0005, momentum=0.937,
        patience=20, warmup_epochs=5, close_mosaic=15,
        project='/kaggle/working/weld_defect_detection',
        name='yolo11s_baseline', exist_ok=True,
        plots=True, save=True, verbose=True,
    )
else:
    print("Skipping Model A training (TRAIN_BASELINE=False)")
    print("Set TRAIN_BASELINE=True to train the baseline for comparison.")

Skipping Model A training (TRAIN_BASELINE=False)
Set TRAIN_BASELINE=True to train the baseline for comparison.


#### ============================================================
# CELL 10 — FULL TRAINING PIPELINE: CRAG + YOLOv11 (DUAL GPU)
# ============================================================
# Architecture : YOLOv11s (C2PSA attention, deeper neck, 
#                improved anchor-free head vs YOLOv8)
# Preprocessor : CRAG-Net (Sobel gradient + Laplacian texture
#                fusion via channel attention)
# Hardware     : Dual NVIDIA T4 via DataParallel
# Classes      : Crack | Porosity | Spatters | Welding line
# ============================================================

!pip install ultralytics kornia pyyaml --quiet

import os
import yaml
import torch
import torch.nn as nn
import kornia
from ultralytics import YOLO
from pathlib import Path

# ── Reproducibility ──────────────────────────────────────────
import random, numpy as np
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# ── GPU Setup (Dual T4) ───────────────────────────────────────
device   = 'cuda' if torch.cuda.is_available() else 'cpu'
num_gpus = torch.cuda.device_count()
print("=" * 60)
print("  HARDWARE CONFIGURATION")
print("=" * 60)
print(f"  Device : {device}")
print(f"  GPUs   : {num_gpus}")
for i in range(num_gpus):
    free, total = torch.cuda.mem_get_info(i)
    print(f"  GPU {i}  : {torch.cuda.get_device_name(i)} "
          f"| {free/1e9:.1f} GB free / {total/1e9:.1f} GB total")
torch.backends.cudnn.benchmark = True

# ── Class Definitions ─────────────────────────────────────────
# These must match the integer label IDs in your .txt annotations:
#   0 → Crack  |  1 → Porosity  |  2 → Spatters  |  3 → Welding line
CLASS_NAMES = ['Crack', 'Porosity', 'Spatters', 'Welding line']
NUM_CLASSES  = len(CLASS_NAMES)

print("\n" + "=" * 60)
print("  DATASET CLASSES")
print("=" * 60)
for idx, name in enumerate(CLASS_NAMES):
    print(f"  {idx} → {name}")

# ── Dataset Root ──────────────────────────────────────────────
DATASET_ROOT = "/kaggle/input/datasets/josephinenakalembe/weld-defect-detection"

# Handle nested Kaggle folder structure
subfolders = os.listdir(DATASET_ROOT)
if len(subfolders) == 1 and os.path.isdir(os.path.join(DATASET_ROOT, subfolders[0])):
    DATASET_ROOT = os.path.join(DATASET_ROOT, subfolders[0])
print(f"\nDataset root: {DATASET_ROOT}")

# ── Auto-detect train / val paths ────────────────────────────
def find_images_path(base, keyword):
    for root, dirs, files in os.walk(base):
        if keyword in root.lower() and "images" in root.lower():
            return root
    return None

train_path = find_images_path(DATASET_ROOT, "train")
val_path   = find_images_path(DATASET_ROOT, "val") or find_images_path(DATASET_ROOT, "valid")

if train_path is None or val_path is None:
    raise FileNotFoundError("Could not locate train/val image directories. "
                            "Check DATASET_ROOT structure.")

print(f"  Train images : {train_path}")
print(f"  Val   images : {val_path}")

# ── Verify label IDs match CLASS_NAMES ───────────────────────
def verify_label_ids(label_dir, expected_nc):
    found_ids = set()
    for lf in Path(label_dir).glob("*.txt"):
        with open(lf) as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    found_ids.add(int(parts[0]))
    unexpected = {i for i in found_ids if i >= expected_nc}
    if unexpected:
        print(f"  WARNING: label IDs {unexpected} exceed nc={expected_nc}. "
              f"Review annotations.")
    else:
        print(f"  Label IDs verified — all within [0, {expected_nc - 1}]")

train_label_dir = train_path.replace("images", "labels")
verify_label_ids(train_label_dir, NUM_CLASSES)

# ── Write YAML ────────────────────────────────────────────────
yaml_path = "/kaggle/working/weld_defect_yolov11.yaml"

with open(yaml_path, "w") as f:
    yaml.dump({
        "train" : train_path,
        "val"   : val_path,
        "nc"    : NUM_CLASSES,
        "names" : CLASS_NAMES,
    }, f, default_flow_style=False, sort_keys=False)

print(f"\n  YAML written → {yaml_path}")

# ── CRAG-Net Preprocessor (Dual-GPU compatible) ───────────────
# Lightweight version for end-to-end training:
#   Stage 1 — Gradient map  (Sobel edge detector)
#           — Texture map   (Laplacian of Gaussian)
#   Stage 2 — Channel attention weights (1 × 1 × 3)
#   Stage 3 — Weighted fusion → 3-channel enhanced image
#
# Using channel-global attention (AdaptiveAvgPool2d) keeps the
# module parameter-free relative to spatial resolution, so it
# scales to any input size without modification.

class PseudoModalityGenerator(nn.Module):
    """Stage 1: Produce gradient and texture modalities from RGB."""
    def forward(self, x: torch.Tensor):
        grad    = kornia.filters.sobel(x)          # edges / cracks
        texture = kornia.filters.laplacian(x, 3)   # pores / spatters
        return x, grad, texture


class CRAGFusion(nn.Module):
    """
    Stage 2+3: Cross-Representation Attention Gate Fusion.
    
    Learns per-class-optimal blending weights [W_rgb, W_grad, W_tex]
    such that:
      Crack        → high W_grad   (sharp linear edges)
      Porosity     → high W_tex    (irregular pit texture)
      Spatters     → balanced      (mixed signature)
      Welding line → high W_rgb    (colour / spatial context)
    """
    def __init__(self, channels: int = 3):
        super().__init__()
        # Global context → 3 scalar weights, one per modality
        self.attention = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels * 3, channels * 3 // 2, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels * 3 // 2, 3, kernel_size=1),
            nn.Softmax(dim=1),          # W_rgb + W_grad + W_tex = 1
        )
        # Post-fusion refinement
        self.refine = nn.Conv2d(channels, channels, kernel_size=1)

    def forward(self, rgb, grad, texture):
        combined        = torch.cat([rgb, grad, texture], dim=1)
        weights         = self.attention(combined)          # (B, 3, 1, 1)
        W_rgb, W_g, W_t = (weights[:, i:i+1] for i in range(3))
        fused           = W_rgb * rgb + W_g * grad + W_t * texture
        return self.refine(fused)


class CRAGNet(nn.Module):
    """Complete CRAG front-end: RGB → CRAG-enhanced RGB."""
    def __init__(self):
        super().__init__()
        self.modality_gen = PseudoModalityGenerator()
        self.fusion       = CRAGFusion(channels=3)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        rgb, grad, texture = self.modality_gen(x)
        return self.fusion(rgb, grad, texture)


# Instantiate and wrap for multi-GPU
crag_net = CRAGNet().to(device)
if num_gpus > 1:
    crag_net = nn.DataParallel(crag_net)
    print(f"\n  CRAG-Net: DataParallel across {num_gpus} GPUs")
crag_net.eval()

crag_params = sum(p.numel() for p in crag_net.parameters())
print(f"  CRAG-Net parameters: {crag_params:,}")


# ── Callback: apply CRAG preprocessing before each batch ─────
def on_preprocess_batch(batch):
    """Inject CRAG-enhanced images into the training/val batch."""
    imgs = batch['img'].to(device).float() / 255.0
    with torch.no_grad():
        enhanced = crag_net(imgs)
    batch['img'] = (enhanced * 255.0).clamp(0.0, 255.0)
    return batch


# ── Training Helper ───────────────────────────────────────────
COMMON_TRAIN_ARGS = dict(
    data         = yaml_path,
    epochs       = 100,
    patience     = 30,
    imgsz        = 640,
    batch        = 32,          # 16 per GPU on dual-T4
    device       = list(range(num_gpus)) if num_gpus > 1 else 0,
    amp          = True,
    workers      = 8,
    cache        = True,
    optimizer    = 'AdamW',
    lr0          = 0.001,
    lrf          = 0.01,
    cos_lr       = True,
    weight_decay = 0.0005,
    momentum     = 0.937,
    warmup_epochs= 5,
    close_mosaic = 15,
    hsv_h        = 0.015,
    hsv_s        = 0.7,
    hsv_v        = 0.4,
    degrees      = 10,
    translate    = 0.1,
    scale        = 0.5,
    shear        = 2.0,
    fliplr       = 0.5,
    mosaic       = 1.0,
    mixup        = 0.1,
    copy_paste   = 0.1,
    project      = '/kaggle/working/weld_defect_detection',
    plots        = True,
    save         = True,
    verbose      = True,
)

# ── MODEL A: CRAG + YOLOv11s ─────────────────────────────────
print("\n" + "=" * 60)
print("  MODEL A: CRAG-YOLOv11s — Weld Defect Detection")
print("=" * 60)
# YOLOv11s key improvements over YOLOv8:
#   • C2PSA (Cross-Stage Partial with Spatial Attention) blocks
#   • Deeper P3/P4/P5 neck with enhanced feature aggregation
#   • Anchor-free decoupled head with improved small-object recall
#   • ~9 % fewer parameters than YOLOv8s at equal mAP

model_crag = YOLO("yolo11s.pt")
model_crag.add_callback("on_preprocess_batch", on_preprocess_batch)

model_crag.train(
    **COMMON_TRAIN_ARGS,
    name = "crag_yolov11s_weld_defect",
)

# ── MODEL B: Baseline YOLOv11s ───────────────────────────────
print("\n" + "=" * 60)
print("  MODEL B: Baseline YOLOv11s — Weld Defect Detection")
print("=" * 60)

model_base = YOLO("yolo11s.pt")

model_base.train(
    **COMMON_TRAIN_ARGS,
    name = "baseline_yolov11s_weld_defect",
)

# ── Summary ───────────────────────────────────────────────────
print("\n" + "=" * 60)
print("  TRAINING COMPLETE")
print("=" * 60)
print("  Results saved to /kaggle/working/weld_defect_detection/")
print("  CRAG model  → crag_yolov11s_weld_defect/weights/best.pt")
print("  Baseline    → baseline_yolov11s_weld_defect/weights/best.pt")
print("  Compare mAP50 / mAP50-95 in the results.csv files.")
print("=" * 60)
